# 02. Tokenizer 元件：從文字到模型輸入 (2026 版)

> 模組：`01-Component / 02tokenizer` ｜ 對應檔案：`02.tokenizer.ipynb`

Tokenizer 是 NLP/LLM 流程中「文字 ↔ 數字」的橋樑。任何模型都看不懂字串，它只吃 token id 張量。這個 notebook 帶你從零理解一段中文／英文如何被切詞、編碼、填充，最後變成模型能直接吃進去的 `input_ids` / `attention_mask`。

## 學習目標

完成本 notebook 後，你應該能：

1. 用 `AutoTokenizer.from_pretrained()` 載入並在本地儲存 tokenizer。
2. 理解 **tokenize → convert_tokens_to_ids → decode** 的完整往返流程。
3. 正確使用 **padding / truncation / return_tensors** 產出可餵入模型的 batch。
4. 看懂 `attention_mask`、`token_type_ids` 的用途。
5. 使用 **`apply_chat_template()`** 處理對話資料（2026 LLM 主流做法）。
6. 區分 **Fast / Slow tokenizer**，並用 `offset_mapping` / `word_ids()` 對齊原始字元（NER 必備）。
7. 知道 tokenizer 如何銜接 **`AutoProcessor`**（多模態）。

## 前置知識

- Python 基礎、對張量（tensor）有概念。
- 已看過 [`../01pipeline/01.pipeline.ipynb`](../01pipeline/01.pipeline.ipynb)：pipeline 內部其實就是先呼叫 tokenizer 再呼叫 model。

## 課程銜接

- 上一站：[`../01pipeline/01.pipeline.ipynb`](../01pipeline/01.pipeline.ipynb) — 高階一鍵推論。
- 下一站：[`../03Model/`](../03Model/) — 把 tokenizer 的輸出餵進模型。
- 多模態延伸：[`../../05-Multimodal/`](../../05-Multimodal/) — 影像/語音用 `AutoProcessor`，與本章的 tokenizer 是同一套設計哲學。

## 版本鎖定 (Reproducibility)

> **WHY**：2024 年很多教材直接 `!pip install transformers` 不鎖版本，結果隔半年 API 行為飄移、範例跑不動。2026 的統一慣例是**把版本鎖定 cell 放最前面**，確保任何人 clone 下來都能重現。

`tokenizers>=0.20` 才有穩定的 fast tokenizer 與 offset mapping 行為；`datasets>=3.0` 與後面的 batch 處理範例相容。

In [ ]:
# Pin versions for reproducibility (run once per environment)
%pip install -q \
    "transformers>=4.46" \
    "datasets>=3.0" \
    "tokenizers>=0.20" \
    "sentencepiece>=0.2.0"

# sentencepiece is required by some non-BERT tokenizers (e.g. multilingual / LLM models)

In [ ]:
import transformers
import tokenizers

# Quick sanity check that the pinned versions are actually loaded
print("transformers:", transformers.__version__)
print("tokenizers :", tokenizers.__version__)

## Step 1. 載入分詞器 (AutoTokenizer)

> **WHY 用 `AutoTokenizer`**：你不需要記住某個模型用的是 BertTokenizer 還是 LlamaTokenizer。`AutoTokenizer` 會讀取模型的 config 自動挑對正確的類別，並且**預設回傳 Fast 版本**（Rust 實作，速度快很多）。

本章沿用原教材的範例模型 `uer/roberta-base-finetuned-dianping-chinese`（一個中文 RoBERTa，詞表 21128），方便觀察中文逐字切詞的行為。

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "uer/roberta-base-finetuned-dianping-chinese"

# A demo sentence we will reuse throughout the notebook
sen = "安安你好, 幾歲住哪裡"

In [ ]:
# use_fast=True is the 2026 default; we state it explicitly for clarity
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer

### 怎麼讀 tokenizer 的 repr？

印出 tokenizer 時會看到幾個關鍵欄位，逐一說明：

- **`name_or_path`**：來源模型名稱或本地路徑。
- **`vocab_size`**：詞表大小（這裡 21128），代表能辨識多少個不同 token。
- **`model_max_length`**：模型可接受的最長輸入。若顯示一個極大的數字，是保守預設值，**請以模型實際限制為準**（如 BERT 為 512）。
- **`is_fast`**：`True` 表示這是 Rust 實作的 Fast tokenizer。
- **`padding_side` / `truncation_side`**：填充與截斷發生在序列的哪一側（`right` 表右側）。
- **`special_tokens`**：特殊 token，如 `[CLS]`（分類起點）、`[SEP]`（分隔）、`[PAD]`（填充）、`[UNK]`（未知）。
- **`added_tokens_decoder`**：id ↔ 特殊 token 的對照表，例如 `0 -> [PAD]`、`100 -> [UNK]`。

## Step 2. 在本地儲存與重新載入

> **WHY**：訓練／部署時常需要把 tokenizer 跟著模型一起打包，避免每次都連網下載，也保證版本固定。`save_pretrained` 會寫出 `tokenizer.json`、`vocab.txt`、`special_tokens_map.json` 等檔案。

In [ ]:
# Save tokenizer files to a local directory
tokenizer.save_pretrained("./roberta_tokenizer")

In [ ]:
# Reload from the local path (works exactly like loading from the Hub)
tokenizer = AutoTokenizer.from_pretrained("./roberta_tokenizer/", use_fast=True)
tokenizer

## Step 3. 句子分詞 (tokenize)

> **WHY**：`tokenize()` 只做「切詞」這一步，不做編碼。觀察它的輸出能幫你理解模型「眼中」的最小單位。中文 RoBERTa 多半是逐字切；英文／LLM 則常見 subword（如 `##ing`、`▁the`）。

注意：`tokenize()` **不會**自動加上 `[CLS]` / `[SEP]` 這類特殊 token——那是 `encode()` / `tokenizer(...)` 才做的事。

In [ ]:
# Split the raw string into tokens (no special tokens, no ids yet)
token_list = tokenizer.tokenize(sen)
token_list

## Step 4. 詞表 (vocabulary)

> **WHY**：詞表是 token 字串 ↔ 整數 id 的對照字典。每個 token 都對應唯一 id，編碼／解碼都靠它。`vocab_size` 決定模型 embedding 層的大小。

In [ ]:
# The full vocab is a dict {token: id}; it can be large, so peek at a few entries
vocab = tokenizer.get_vocab()
print("vocab_size:", tokenizer.vocab_size)

# Show the first 10 items (order is not meaningful, just a sample)
for i, (tok, idx) in enumerate(vocab.items()):
    if i >= 10:
        break
    print(repr(tok), "->", idx)

## Step 5. 索引轉換：token ↔ id ↔ string

> **WHY**：理解這三個方向的轉換，你就掌握了 tokenizer 的核心。實務上你很少手動呼叫它們（直接用 `tokenizer(...)` 即可），但 debug 模型輸出、做 token 對齊時非常有用。
>
> - `convert_tokens_to_ids`：token 序列 → id 序列
> - `convert_ids_to_tokens`：id 序列 → token 序列
> - `convert_tokens_to_string`：token 序列 → 還原字串

In [ ]:
# token -> id
ids = tokenizer.convert_tokens_to_ids(token_list)
ids

In [ ]:
# id -> token
tokens = tokenizer.convert_ids_to_tokens(ids)
tokens

In [ ]:
# token -> string (reconstruct readable text)
str_sen = tokenizer.convert_tokens_to_string(tokens)
str_sen

## Step 6. 序列填充與截斷 (padding / truncation)

> **WHY**：模型一次吃一個 batch，要求 batch 內每條序列**等長**。短的補 `[PAD]`（padding），長的砍掉（truncation）。這是把不定長文字變成矩形張量的關鍵步驟。
>
> 常用參數：
> - `padding="max_length"`：補到 `max_length`。
> - `padding=True`：補到「該 batch 內最長那條」（更省記憶體，動態 padding）。
> - `truncation=True`：超過 `max_length` 就截斷。

In [ ]:
# encode() = tokenize + add special tokens + convert to ids, in one call
# Here we pad to a fixed length of 14 to observe [PAD] (id=0) being appended
ids = tokenizer.encode(
    sen,
    padding="max_length",
    max_length=14,
    truncation=True,  # always pair max_length with truncation to be safe
)
ids

In [ ]:
# decode: ids -> string. Keep special tokens visible so you can see [CLS]/[SEP]/[PAD]
str_sen = tokenizer.decode(ids, skip_special_tokens=False)
str_sen

## Step 7. 模型其餘輸入：attention_mask 與 token_type_ids

> **WHY**：光有 `input_ids` 不夠。模型還需要：
>
> - **`attention_mask`**：哪些位置是真實 token（`1`）、哪些是填充（`0`）。模型會忽略 `0` 的位置，避免 `[PAD]` 干擾注意力計算。
> - **`token_type_ids`**（又稱 segment ids）：BERT 類模型用來區分「句子 A / 句子 B」（如問答、句對任務），單句時全為 `0`。
>
> 下面手動算一次只是為了**理解原理**；實務上絕不手刻，交給 `tokenizer(...)` 一鍵產生（見 Step 8）。

In [ ]:
# Illustrative manual computation (DO NOT do this in real code)
# attention_mask: 1 for real tokens, 0 for [PAD] (id 0)
attention_mask = [1 if idx != 0 else 0 for idx in ids]

# token_type_ids: all zeros for a single sentence
token_type_ids = [0] * len(ids)

print("input_ids     :", ids)
print("attention_mask:", attention_mask)
print("token_type_ids:", token_type_ids)

### 流程總結（手動版）

手動串起來的完整流程是：

1. 下載／載入分詞器
2. 句子分詞 `tokenize`
3. 索引轉換 `convert_tokens_to_ids`
4. 序列填充／截斷 `encode(padding=..., truncation=...)`
5. 組裝訓練輸入 = `input_ids` + `attention_mask` + `token_type_ids`

下一節我們用**一行**取代以上全部。

## Step 8. 直接呼叫 tokenizer（推薦做法）

> **WHY**：把 tokenizer 物件當函式呼叫 `tokenizer(text, ...)`，會一次回傳 `input_ids` / `token_type_ids` / `attention_mask`，且自動加好特殊 token。**這才是日常正確寫法**，前面的手動步驟只是教學拆解。
>
> **2026 重點：`return_tensors`**。要餵進 PyTorch 模型時加 `return_tensors="pt"`，直接拿到張量，省掉手動 `torch.tensor(...)`。

In [ ]:
# One-shot encoding -> a BatchEncoding dict with all model inputs
inputs = tokenizer(
    sen,
    padding="max_length",
    max_length=14,
    truncation=True,
    return_tensors="pt",  # "pt" = PyTorch tensors; use "np"/"tf" if needed
)
inputs

## Step 9. 批次處理 (batch encoding)

> **WHY**：實務上你不會一句一句餵。把一個 `list[str]` 丟給 tokenizer，它會一次處理整個 batch。搭配 `padding=True`（動態補到 batch 內最長）最省資源。Fast tokenizer 的批次處理會走 Rust 平行化，遠快於 Python 迴圈逐句處理。

In [ ]:
# A small batch of sentences (space-themed, following the original example)
space_story = [
    "一艘宇宙飛船正在穿越星際，探索未知的星球。",
    "太空站上的科學家們正在研究外星生命的跡象。",
]

# padding=True -> pad to the longest sequence within this batch (dynamic padding)
res = tokenizer(space_story, padding=True, truncation=True, return_tensors="pt")

for k, v in res.items():
    print(k)
    print(v)
    print()

### 效能對照：逐句迴圈 vs. 批次呼叫

> **WHY**：很多人習慣寫 `for` 迴圈逐句 tokenize，這在 Fast tokenizer 下會浪費掉 Rust 的平行能力。比較兩者耗時，建立「永遠優先批次」的直覺。

In [ ]:
%%time
# Slow pattern: call tokenizer once per sentence in a Python loop
for _ in range(1000):
    tokenizer(space_story)

In [ ]:
%%time
# Fast pattern: hand the whole list to the tokenizer in one call
res = tokenizer(space_story * 1000)

## Step 10. 對話資料前處理：`apply_chat_template()`（2026 主流）

> **WHY**：2024 教材幾乎都還在手動拼 `[CLS] ... [SEP]` 或自己接 prompt 字串。但 2026 的 LLM（Llama 3、Qwen、Mistral 等）都自帶 **chat template**——每個模型的對話格式（角色標記、特殊 token）不同，**手刻 prompt 等於踩雷**。
>
> 正確做法是給一個 `messages` 串列（`role` + `content`），交給 `apply_chat_template()` 自動套用該模型專屬格式。
>
> - `tokenize=False`：先看渲染出來的純文字 prompt（debug 用）。
> - `tokenize=True, return_tensors="pt"`：直接拿到可餵入模型的張量。
> - `add_generation_prompt=True`：在結尾補上「助理開始回答」的標記，做生成時必加。

In [ ]:
# Load a chat-style tokenizer that ships a chat template.
# (Swap to any instruct model you have access to, e.g. Qwen / Llama-3-Instruct.)
CHAT_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL_ID, use_fast=True)

messages = [
    {"role": "system", "content": "你是一個樂於助人的太空科普助理。"},
    {"role": "user", "content": "請用一句話解釋什麼是黑洞。"},
]

In [ ]:
# 1) Render to plain text first to SEE the model-specific format (roles, special tokens)
prompt_text = chat_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # append the "assistant turn starts here" marker
)
print(prompt_text)

In [ ]:
# 2) Tokenize directly into tensors, ready to feed model.generate(**chat_inputs)
chat_inputs = chat_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
)
chat_inputs

## Step 11. Fast vs. Slow tokenizer

> **WHY**：`AutoTokenizer` 預設給 Fast 版（Rust，`is_fast=True`）。Slow 版是純 Python 後備，速度慢且**沒有** `offset_mapping`、`word_ids()` 等對齊功能。除非某模型只提供 slow 版，否則永遠用 fast。下面量化兩者差距。

In [ ]:
en_sen = "Hello world, today is python day"

fast_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
slow_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)

print("fast is_fast:", fast_tokenizer.is_fast)
print("slow is_fast:", slow_tokenizer.is_fast)

In [ ]:
%%time
# Fast tokenizer, single-sentence loop
for _ in range(10000):
    fast_tokenizer(en_sen)

In [ ]:
%%time
# Slow tokenizer, single-sentence loop (expect this to be noticeably slower)
for _ in range(10000):
    slow_tokenizer(en_sen)

In [ ]:
%%time
# Fast tokenizer, batch call -> this is where Rust parallelism really shines
res = fast_tokenizer([en_sen] * 10000)

In [ ]:
%%time
# Slow tokenizer, batch call (no real parallelism; much slower)
res = slow_tokenizer([en_sen] * 10000)

## Step 12. offset_mapping 與 word_ids()：對齊回原始字元

> **WHY（NER／抽取式 QA 必備）**：當一個詞被切成多個 subword 時，你需要知道「每個 token 對應原文的哪一段字元」才能把模型的 token 級預測映射回原始文字（例如標出人名的起訖位置）。這只有 **Fast tokenizer** 才支援。
>
> - `return_offsets_mapping=True`：回傳每個 token 的 `(start, end)` 字元區間。
> - `word_ids()`：回傳每個 token 屬於第幾個原始詞（subword 會共享同一個 word id）。

In [ ]:
# Fast tokenizer can return character-level offsets for each token
inputs = fast_tokenizer(en_sen, return_offsets_mapping=True)
inputs

In [ ]:
# The token ids produced for this sentence
inputs.input_ids

In [ ]:
# word_ids(): maps each token back to its source word index (None for special tokens)
inputs.word_ids()

In [ ]:
# Practical demo: slice the original string using offset_mapping
offsets = inputs["offset_mapping"]
toks = fast_tokenizer.convert_ids_to_tokens(inputs["input_ids"])

for tok, (start, end) in zip(toks, offsets):
    # (0, 0) marks special tokens like [CLS]/[SEP] that have no original span
    span = en_sen[start:end] if (start, end) != (0, 0) else "<special>"
    print(f"{tok:>8} -> chars[{start}:{end}] = {span!r}")

## Step 13. 不同架構的 tokenizer 差異

> **WHY**：不同模型家族的 tokenizer 設計不同：
>
> - **BERT / RoBERTa（編碼器）**：有 `token_type_ids`，靠 `[CLS]`/`[SEP]` 處理句對任務。
> - **GPT / Llama / Qwen（解碼器、LLM）**：通常**沒有** `token_type_ids`，改用位置資訊（`position_ids`）處理連續文本流，因為它們是自回歸生成。
>
> 觀察同一句話在兩種 tokenizer 下回傳的 key 差異，是理解模型架構的好切入點。原教材用 ChatGLM2 示範這點；2026 我們用前面已載入的 Qwen instruct tokenizer 做對照（更現代、無需 `trust_remote_code`）。

In [ ]:
# Encoder-style (BERT/RoBERTa): note token_type_ids is present
enc_out = tokenizer(en_sen)
print("RoBERTa keys:", list(enc_out.keys()))

# Decoder/LLM-style (Qwen): typically NO token_type_ids
llm_out = chat_tokenizer(en_sen)
print("Qwen    keys:", list(llm_out.keys()))

In [ ]:
# Round-trip check on the LLM tokenizer: encode then decode should recover the text
encoded = chat_tokenizer.encode(en_sen)
decoded = chat_tokenizer.decode(encoded)
print("encoded:", encoded)
print("decoded:", decoded)

> **2026 安全提醒：`trust_remote_code`**
>
> 舊教材常見 `AutoTokenizer.from_pretrained(..., trust_remote_code=True)`。這會**執行模型 repo 內的任意 Python 程式碼**，是供應鏈風險。2026 慣例：
>
> 1. 優先選用已整合進 `transformers` 主線、不需要 remote code 的模型（如 Qwen2.5、Llama 3）。
> 2. 真的必須開啟時，務必確認來源可信，並**鎖定 commit hash**（`revision="<sha>"`）而非用浮動的 `main`。

## Step 14. 銜接多模態：`AutoProcessor`

> **WHY**：tokenizer 只處理文字。當輸入是「影像 + 文字」「語音 + 文字」時，HuggingFace 用 **`AutoProcessor`**，它在內部把 tokenizer（處理文字）與 image/feature extractor（處理像素／音訊）包在一起，介面與 tokenizer 高度一致——一樣 `from_pretrained` 載入、一樣 `return_tensors="pt"`。
>
> 也就是說，你在本章學到的 padding / truncation / return_tensors 觀念，會原封不動延伸到多模態。深入範例見 [`../../05-Multimodal/`](../../05-Multimodal/)（CLIP、VLM VQA、Whisper ASR 等）。
>
> 下面僅示範 API 形狀（多模態模型較大，視環境決定是否實跑）。

In [ ]:
# Conceptual bridge to multimodal: AutoProcessor wraps a tokenizer + image processor.
# Uncomment to try (downloads a vision-language model).
#
# from transformers import AutoProcessor
# from PIL import Image
#
# processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
# image = Image.open("some_image.jpg")
# mm_inputs = processor(
#     text=["a photo of a spaceship", "a photo of a cat"],
#     images=image,
#     return_tensors="pt",
#     padding=True,
# )
# print(mm_inputs.keys())  # e.g. input_ids, attention_mask, pixel_values
#
# The text path inside processor IS a tokenizer:
# print(type(processor.tokenizer))
print("See ../../05-Multimodal/ for full multimodal examples (CLIP / VLM / Whisper).")

## 小結 (Summary)

本章從原理到實務，串起了 tokenizer 的完整生命週期：

| 主題 | 關鍵 API / 觀念 |
| --- | --- |
| 載入／儲存 | `AutoTokenizer.from_pretrained(..., use_fast=True)`、`save_pretrained` |
| 切詞往返 | `tokenize` → `convert_tokens_to_ids` → `decode` |
| 模型輸入 | `tokenizer(text, padding, truncation, return_tensors="pt")` |
| 批次效能 | 一律批次呼叫，善用 Fast tokenizer 的 Rust 平行化 |
| 對話前處理 | `apply_chat_template(messages, add_generation_prompt=True)` |
| 字元對齊 | Fast 限定的 `return_offsets_mapping` / `word_ids()` |
| 架構差異 | 編碼器有 `token_type_ids`；LLM 多半沒有 |
| 多模態銜接 | `AutoProcessor` = tokenizer + image/feature extractor |

**核心心法**：日常只需要 `tokenizer(text, ...)` 一行；手動拆步驟只是為了理解內部發生什麼。

## 練習 (Exercises)

1. **動態 padding**：把 `space_story` 用 `padding=True` 與 `padding="max_length", max_length=64` 各編碼一次，比較兩者 `input_ids` 的形狀與 `[PAD]` 數量，說明哪種較省記憶體。
2. **截斷觀察**：造一個超過 `max_length` 的長句，設定 `truncation=True, max_length=10`，觀察 `[SEP]` 是否仍保留在結尾、被砍掉的是哪部分。
3. **chat template**：在 `messages` 中加入第二輪 `user`/`assistant` 對話，比較 `add_generation_prompt=True` 與 `False` 渲染出的文字差異。
4. **NER 對齊**：用 Fast tokenizer 對一句含人名的英文句子取 `offset_mapping`，寫一個函式：給定原文的字元區間 `(start, end)`，回傳涵蓋它的 token id 清單。
5. **架構比較**：再載入一個 BERT 與一個 LLM tokenizer，對同一句話編碼，列出兩者回傳 key 的差集，並解釋為何 LLM 不需要 `token_type_ids`。
6. **（挑戰）多模態**：到 [`../../05-Multimodal/`](../../05-Multimodal/) 找一個 `AutoProcessor` 範例，確認其 `processor.tokenizer` 的型別，並對照本章學到的 `return_tensors` / `padding` 行為是否一致。